In [1]:
import torch
import torchvision

import torch
import torch.nn as nn

from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader, random_split


from torchvision import transforms

from torchvision.transforms import v2

#################################################################
from sklearn.model_selection import KFold
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


from matplotlib.ticker import AutoMinorLocator, MultipleLocator

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder

from sklearn.utils import check_random_state

import os
import json
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from pathlib import Path

import pandas as pd
import mlflow
from mlflow import MlflowClient

plt.style.use('dark_background')

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [2]:
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("House_Prices_DL_Kaggle")

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1792261869877, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1792261869877, lifecycle_stage='active', name='House_Prices_DL_Kaggle', tags={}, trace_location=None, workspace='default'>

In [5]:
X_cols = [
'MSSubClass',
 'MSZoning',
 'Alley',
 'LotConfig',
 'LandContour',
 'Neighborhood',
 'BldgType',
 'HouseStyle',
 'RoofStyle',
 'RoofMatl',
 'Exterior1st',
 'Foundation',
 'Heating',
 'Condition1',
 'Condition2',
 'Exterior2nd',
 'GarageType',
 'PavedDrive',
 'Fence',
 'MiscFeature',
 'SaleType',
 'SaleCondition',
 'LotShape',
 'Electrical',
 'FullBath',
 'TotRmsAbvGrd',
 'GarageCars',
 'BedroomAbvGr',
 'KitchenAbvGr',
 'Utilities',
 'OverallQual',
 'OverallCond',
 'ExterCond',
 'BsmtQual',
 'HeatingQC',
 'KitchenQual',
 'GarageFinish',
 'Functional',
 'LandSlope',
 'BsmtExposure',
 'CentralAir',
 'Street',
 'BsmtHalfBath',
 'HalfBath',
 'OpenPorch',
 'EnclosedPorch',
 '3SsnPorch',
 'ScreenPorch',
 'LowQualFin',
 'BsmtFin',
 'LotFrontage',
 'LotArea',
 'MasVnrArea_log',
 'BsmtFinSF1_log',
 'BsmtUnfSF_log',
 'TotalBsmtSF_log',
 '1stFlrSF',
 '2ndFlrSF_log',
 'GarageArea_log',
 'WoodDeckSF',
 'OpenPorchSF',
 'MiscVal',
 'TotalSqFeet',
 'MoSold',
 'YrSold',
 'Age',
 'Age_from_Remod',
 'OverallQual*OverallCond',
 'ExterQual*ExterCond',
 'BsmtFinType1*BsmtFinSF1*BsmtQual',
 'BsmtFinType2*BsmtFinSF2*BsmtQual',
 'OverallBath',
 'PoolOverall',
 'Overall_Fireplaces']

In [6]:
num_cols = ['1stFlrSF',
 '2ndFlrSF_log',
 '3SsnPorch',
 'Age',
 'Age_from_Remod',
 'BedroomAbvGr',
 'BsmtExposure',
 'BsmtFin',
 'BsmtFinSF1_log',
 'BsmtFinType1*BsmtFinSF1*BsmtQual',
 'BsmtFinType2*BsmtFinSF2*BsmtQual',
 'BsmtHalfBath',
 'BsmtQual',
 'BsmtUnfSF_log',
 'CentralAir',
 'EnclosedPorch',
 'ExterCond',
 'ExterQual*ExterCond',
 'FullBath',
 'Functional',
 'GarageArea_log',
 'GarageCars',
 'GarageFinish',
 'HalfBath',
 'HeatingQC',
 'KitchenAbvGr',
 'KitchenQual',
 'LandSlope',
 'LotArea',
 'LotFrontage',
 'LowQualFin',
 'MasVnrArea_log',
 'MiscVal',
 'MoSold',
 'OpenPorch',
 'OpenPorchSF',
 'OverallBath',
 'OverallCond',
 'OverallQual',
 'OverallQual*OverallCond',
 'Overall_Fireplaces',
 'PoolOverall',
 'ScreenPorch',
 'Street',
 'TotRmsAbvGrd',
 'TotalBsmtSF_log',
 'TotalSqFeet',
 'Utilities',
 'WoodDeckSF',
 'YrSold']

In [7]:
cat_cols = ['MSSubClass',
 'MSZoning',
 'Alley',
 'LotConfig',
 'LandContour',
 'Neighborhood',
 'BldgType',
 'HouseStyle',
 'RoofStyle',
 'RoofMatl',
 'Exterior1st',
 'Foundation',
 'Heating',
 'Condition1',
 'Condition2',
 'Exterior2nd',
 'GarageType',
 'PavedDrive',
 'Fence',
 'MiscFeature',
 'SaleType',
 'SaleCondition',
 'LotShape',
 'Electrical']

In [8]:
data = pd.read_csv('train_data.csv')

## Рисовашки

In [9]:
def draw_live_metrics(train_RMSE, val_RMSE, fold):

    clear_output(wait=True)

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(
        range(1, len(train_RMSE) + 1),
        train_RMSE,
        label='Train RMSE'
    )

    ax.plot(
        range(1, len(val_RMSE) + 1),
        val_RMSE,
        label='Val RMSE'
    )

    best_epoch = np.argmin(val_RMSE) + 1
    best_score = np.min(val_RMSE)

    ax.scatter(
        best_epoch,
        best_score,
        color='red',
        zorder=5,
        label=f'Best RMSE = {best_score:.5f}'
    )

    ax.set_title(f'Fold {fold} | Epoch {len(val_RMSE)}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('RMSE')

    ax.legend()
    ax.grid(alpha=0.3)

    display(fig)
    plt.close(fig)

## Классы для обучения

In [10]:
class EarlyStopping:
    def __init__(self,mode = 'min',patience = 10, threshold = 0.001, threshold_mode = 'abs'):
        self.mode = mode
        self.patience = patience
        self.threshold = threshold
        self.threshold_mode = threshold_mode
        self.count = 0
        self.best = None

    def __call__(self, trecked_param):
        current = float(trecked_param)

        if self.best is None:
            self.best = current
            return False

        if self.changed_better(current, self.best):
            self.best = current
            self.count = 0
        else:
            self.count += 1

        if self.count>=self.patience:
            self.count = 0
            return True
        return False

    def changed_better(self,current,best):
        return current < best - self.threshold

In [11]:
def conf_model(n_num,cat_num,emb_din,inner_n,optim_lr,drop_n,factor,patience,threshold,min_lr,early_stopping_patience,early_stopping_threshold,n_layers,use_batchnorm):
    #Создаем модель
    model_reg = MyModel(cat_num,n_num,emb_din,inner_n,drop_n,n_layers,use_batchnorm).to(device)

    #Определяем функцию потерь и оптимизатор градиентного спуска
    loss_reg = nn.MSELoss()
    opt_reg = torch.optim.Adam(model_reg.parameters(), lr = optim_lr)
    
    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_reg,mode='min',factor=factor,patience=patience,threshold=threshold,min_lr=min_lr,threshold_mode='abs')
    early_stopping = EarlyStopping(mode = 'min',patience = early_stopping_patience, threshold = early_stopping_threshold, threshold_mode = 'abs')

    return model_reg,loss_reg,opt_reg,lr_scheduler,early_stopping

In [12]:
def transform_data(X_train,X_val,y_train,y_val,cat_cols, num_cols,batch_size):
    #Конфигурируем преобразователи
    encoder = OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1,
        encoded_missing_value=-1)
    scaler = StandardScaler()
    
    #Разделяем на категориальные и числовые признаки
    X_train_cat = X_train[cat_cols]
    X_train_num = X_train[num_cols]

    X_val_cat = X_val[cat_cols]
    X_val_num = X_val[num_cols]

    #Кодируем категориальные признаки
    X_train_transformed = encoder.fit_transform(X_train_cat).astype('int64') + 1
    X_val_transformed = encoder.transform(X_val_cat).astype('int64') + 1

    #Считаем кол-во строк для эмбединга каждого признака
    cat_num = [
        len(categories) + 1
        for categories in encoder.categories_]
    
    #Нормализуем числовые фичи
    X_train_scaled = scaler.fit_transform(X_train_num)
    X_val_scaled = scaler.transform(X_val_num)

    #Считаем кол-во строк матрицы эмбедингов
    n_num = X_train_num.shape[1]
        
    #Создаем датасет и даталоадер
    train_data = TableDataset(X_cat = X_train_transformed, X_num = X_train_scaled, y = y_train.values)
    val_data = TableDataset(X_cat = X_val_transformed, X_num = X_val_scaled, y = y_val.values)

    train_loader = DataLoader(train_data,batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data,batch_size=batch_size, shuffle=True)

    return train_loader,val_loader,n_num,cat_num,encoder,scaler

In [13]:
class TableDataset(Dataset):    
    def __init__(self, X_cat, X_num, y):
        self.X_cat = torch.tensor(
            X_cat,
            dtype=torch.long
        )

        self.X_num = torch.tensor(
            X_num,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        ).reshape(-1)
        
    def __len__(self):
        # Возвращает общее количество строк в таблице
        return len(self.X_num)
    
    def __getitem__(self, idx):
        # Возвращает одну строку (признаки и таргет) по индексу
        return self.X_num[idx], self.X_cat[idx], self.y[idx]

In [14]:
def run_epoch(model, train_loader, val_loader,loss_reg, opt_reg,flag='train_val'):
    train_loss = []
    train_RMSE = []
    val_loss = []
    val_RMSE = []

    if flag in ('train_val', 'train'):
        model.train()

        loss_sum = 0.0
        total_samples = 0

        for x_num, x_cat, targets in train_loader:
            x_num = x_num.to(device)
            x_cat = x_cat.to(device)
            targets = targets.to(device)

            opt_reg.zero_grad()

            pred = model(x_num,x_cat)
            loss = loss_reg(pred, targets)

            loss.backward()
            opt_reg.step()

            batch_size = targets.size(0)

            loss_sum += loss.item() * batch_size
            total_samples += batch_size

        train_loss.append(loss_sum / total_samples)
        train_RMSE.append((loss_sum / total_samples) ** 0.5)

    if flag in ('train_val', 'val'):
        model.eval()

        # Отдельные накопители для валидации
        loss_sum = 0.0
        total_samples = 0

        with torch.no_grad():
            for x_num, x_cat, targets in val_loader:
                x_num = x_num.to(device)
                x_cat = x_cat.to(device)
                targets = targets.to(device)

                pred = model(x_num,x_cat)
                loss = loss_reg(pred, targets)

                batch_size = targets.size(0)

                loss_sum += loss.item() * batch_size
                total_samples += batch_size

        val_loss.append(loss_sum / total_samples)
        val_RMSE.append((loss_sum / total_samples) ** 0.5)

    return train_loss, train_RMSE, val_loss, val_RMSE

In [15]:
def save_model(fold,epoch,best_epoch,best_score,mean_val_RMSE,saving_threshold,encoder,scaler,model_reg):

    if mean_val_RMSE < best_score - saving_threshold or epoch == 0:

        best_score = mean_val_RMSE
        best_epoch = epoch + 1

        pipeline_state = {
            'encoder': encoder,
            'scaler': scaler,
            'model': model_reg
        }

        torch.save(
            pipeline_state,
            f"./DL_Models/full_pipeline_epoch--{epoch+1}, fold--{fold}.pt"
        )

    return best_score,best_epoch

## Валидация

In [16]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

## Модель

In [17]:
class MyModel(nn.Module):
    def __init__(self, cat_num, num_num, emb_din, inner_n, drop_n,n_layers,use_batchnorm = False):
        '''
        cat_num - размеры словарей категорий для каждого столбца
        num_num - кол-во числовых колонок
        emb_din - размер эмбедингов
        inner_n - размер внутреннего слоя
        drop_n - процент отключаемых нейронов в слое
        n_layers - кол-во слоев
        '''
        super().__init__()
        #Создаем отдельную матрицу эмбедингов для каждого категориального признака
        embeddings = []
        #эмбединги обучаются так же бэк-пропом
        for n_cat in cat_num:
            layer = nn.Embedding(
                num_embeddings=n_cat,
                embedding_dim=emb_din,
                padding_idx=0
            )
            embeddings.append(layer)

        self.embeddings = nn.ModuleList(embeddings)
        #nn.Embedding — это матрица обучаемых весов
        n_embeddings = sum(
            embedding.embedding_dim
            for embedding in self.embeddings
        )

        input = num_num + n_embeddings

        layers = []
        n_features = input

        for i in range(n_layers):

            layers.append(nn.Linear(n_features, inner_n))

            if use_batchnorm:
                layers.append(nn.BatchNorm1d(inner_n))

            layers.append(nn.ReLU())
            layers.append(nn.Dropout(drop_n))

            n_features = inner_n

        layers.append(nn.Linear(n_features, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        cat_embeddings = []

        #Вытаскиваем нужные нам эмбединги
        for i, embedding in enumerate(self.embeddings):
            category_indexes = x_cat[:, i]
            embedding_vectors = embedding(category_indexes)
            cat_embeddings.append(embedding_vectors)

        #Объединяем по измерению признаков
        x = torch.cat([x_num] + cat_embeddings,dim=1)

        return self.network(x).squeeze(-1) #делает froward pass по всей сети и убирает последнюю ненужную размерность

## Препроцессинг

In [18]:
X = data[X_cols]
y = data['SalePrice_log']

## Цикл обучения

сначала посмотреть как на больших батчах будет, затем снизить фактор

In [ ]:
#Hyperparams
EPOCHS = 250 #Кол-во эпох обучения
#Model
emb_din = 12
inner_n = 128
drop_n = 0.2
n_layers = 2
use_batchnorm = False
#lr_scheduler
factor = 0.9       # Коэффициент уменьшения LR: новый LR = старый LR * factor
patience = 10      # Количество эпох без значимого улучшения метрики до снижения LR
threshold = 0.01 #abs  # Минимальное изменение метрики, считающееся улучшением
min_lr = 0.00000001    # Нижняя граница LR, ниже которой scheduler его не уменьшает
#EarlyStopping
early_stopping_patience = 60 #Кол-во эпох без измененния отслеживаемой переменной
early_stopping_threshold = 0.00001 #Насколько должна измениться отслеживаемая переменная
#Adam
optim_lr = 0.005 #скорость 0.4 может приводить к тому, что оптимизатор проскакивает хорошие области функции потерь, значения предсказаний резко меняются, а в дальнейшем обучение застревает на неудачном плато.
#Dataloader
batch_size = 32 #Размер батча
#Saving threshold 
saving_threshold = 0.001

In [ ]:
#Metrics
train_loss_all_folds = []
train_RMSE_all_folds = []
val_loss_all_folds = []
val_RMSE_all_folds = []
model_list = []

mean_fold_val_loss = []
learning_rate_all_folds = []

#------
best_epochs_for_folds = []
best_scores_for_folds = []

folder_path = Path(r"C:\Users\user\3D Objects\code 2\Mentor_python\Block 2\Final_prodject\House Prices\DL_Models")

# Перебираем все элементы и удаляем только файлы
for item in folder_path.iterdir():
  if item.is_file():
    item.unlink()

with mlflow.start_run(run_name='CV_Embedding_MLP') as parent_run:
    parent_run_id = parent_run.info.run_id

    mlflow.log_params({
        'epochs': EPOCHS,
        'emb_dim': emb_din,
        'inner_n': inner_n,
        'dropout': drop_n,
        'optim_lr': optim_lr,
        'batch_size': batch_size,
        'lr_scheduler_factor': factor,
        'lr_scheduler_patience': patience,
        'lr_scheduler_threshold':threshold,
        'lr_scheduler_min_lr':min_lr,
        'early_stopping_patience': early_stopping_patience,
        'early_stopping_threshold': early_stopping_threshold,
        'saving_threshold': saving_threshold,
        'n_layers':n_layers,
        'n_splits': kf.get_n_splits(),
        'use_batchnorm':use_batchnorm,
        'seed': SEED
    })

    folds = tqdm(
        enumerate(kf.split(X, y), 1),
        total=kf.get_n_splits(),
        desc='Cross Validation',
        position=0)
    
    for fold, (train_idx, val_idx) in folds:
        with mlflow.start_run(run_name=f'Fold_{fold}', nested=True):
            
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            #Преобразования связанные с данными
            train_loader,val_loader,n_num,cat_num,encoder,scaler = transform_data(X_train,X_val,y_train,y_val,cat_cols, num_cols,batch_size)

            #Конфигурация модели
            model_reg,loss_reg,opt_reg,lr_scheduler,early_stopping = conf_model(n_num,cat_num,emb_din,inner_n,optim_lr,drop_n,factor,patience,threshold,min_lr,early_stopping_patience,early_stopping_threshold,n_layers,use_batchnorm)

            epochs_train_loss = []
            epochs_train_RSME = []
            epochs_val_loss = []
            epochs_val_RSME = []

            learning_rate_epoch = []

            best_score = float('inf')
            best_epoch = None

            epochs = tqdm(
                range(EPOCHS),
                desc=f'Fold {fold}',
                leave=False,
                position=1)

            for epoch in epochs:
                current_lr = opt_reg.param_groups[0]['lr']
                learning_rate_epoch.append(current_lr)

                train_loss,train_RMSE,val_loss,val_RMSE = run_epoch(model_reg,train_loader,val_loader,loss_reg,opt_reg)

                train_mse = train_loss[0]
                train_rmse = train_RMSE[0]
                val_mse = val_loss[0]
                val_rmse = val_RMSE[0]

                epochs_train_loss.append(train_mse)
                epochs_train_RSME.append(train_rmse)
                epochs_val_loss.append(val_mse)
                epochs_val_RSME.append(val_rmse)
                
                lr_scheduler.step(val_rmse)

                mlflow.log_metrics({
                        'train_mse': train_mse,
                        'train_rmse': train_rmse,
                        'val_mse': val_mse,
                        'val_rmse': val_rmse,
                        'learning_rate': current_lr
                    }, step=epoch + 1)
                
                best_score,best_epoch = save_model(fold,epoch,best_epoch,best_score,val_rmse,saving_threshold,encoder,scaler,model_reg)
                
                if early_stopping(val_rmse):
                    print(f"\033[31mОбучение остановлено на {epoch + 1} эпохе.\033[0m")
                    break

            
            checkpoint_path = f'./DL_Models/full_pipeline_epoch--{best_epoch}, fold--{fold}.pt'

            mlflow.log_artifact(
                checkpoint_path,
                artifact_path='checkpoint'
            )
            mlflow.log_metrics({
                'best_val_rmse': best_score,
                'best_epoch': best_epoch,
                'epochs_trained': epoch + 1
            })
            mlflow.log_params({
                'fold': fold,
                'n_numeric_features': n_num,
                'n_categorical_features': len(cat_num),
                'n_model_parameters': sum(
                    p.numel() for p in model_reg.parameters()
                )
            })


        #На каждом фолде выбираем скачанную модель по лучшей эпохе и еще раз проверяем ее на валидации
        final_pipeline = torch.load(f"./DL_Models/full_pipeline_epoch--{best_epoch}, fold--{fold}.pt", map_location=device,weights_only = False)
        final_model = final_pipeline['model']

        final_encoded = final_pipeline['encoder']
        final_scaler = final_pipeline['scaler']

        _, _ , best_fold_loss, best_fold_RMSE =run_epoch(final_model,None,val_loader,loss_reg,None,flag='val')
        
        best_epochs_for_folds.append(best_epoch)
        best_scores_for_folds.append(best_fold_RMSE[0])   

        train_loss_all_folds.append(epochs_train_loss)
        train_RMSE_all_folds.append(epochs_train_RSME)
        val_loss_all_folds.append(epochs_val_loss)
        val_RMSE_all_folds.append(epochs_val_RSME)

        learning_rate_all_folds.append(learning_rate_epoch)

    cv_mean = float(np.mean(best_scores_for_folds))
    cv_std = float(np.std(best_scores_for_folds))

    mlflow.log_metrics({
        'cv_mean_rmse': cv_mean,
        'cv_std_rmse': cv_std
    })

    results_df = pd.DataFrame({
        'fold': range(1, len(best_scores_for_folds) + 1),
        'best_epoch': best_epochs_for_folds,
        'best_val_rmse': best_scores_for_folds
    })

    mlflow.log_table(
        results_df,
        artifact_file='cv_results.json'
    )

    print(f'Mean CV RMSE: {cv_mean:.5f}')
    print(f'Std CV RMSE: {cv_std:.5f}')

Cross Validation:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 1:   0%|          | 0/250 [00:00<?, ?it/s]

🏃 View run Fold_1 at: http://127.0.0.1:5000/#/experiments/3/runs/70b1d7012e5040f08dbff92e7ecae93c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


Fold 2:   0%|          | 0/250 [00:00<?, ?it/s]

🏃 View run Fold_2 at: http://127.0.0.1:5000/#/experiments/3/runs/1093151a4fec48dd9a424986690fa704
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


...

Обучение остановлено на 201 эпохе.
🏃 View run Fold_3 at: http://127.0.0.1:5000/#/experiments/3/runs/c9ddd834709e4fbfb325beb0af837c30
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


Fold 4:   0%|          | 0/250 [00:00<?, ?it/s]

🏃 View run Fold_4 at: http://127.0.0.1:5000/#/experiments/3/runs/4f1f555a2e3a4ed79dfab8880ad8452e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


Fold 5:   0%|          | 0/250 [00:00<?, ?it/s]

🏃 View run Fold_5 at: http://127.0.0.1:5000/#/experiments/3/runs/7072393f53d4427193191c506e3c6d3e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Mean CV RMSE: 0.17364
Std CV RMSE: 0.02901
🏃 View run CV_Embedding_MLP at: http://127.0.0.1:5000/#/experiments/3/runs/d413a3d6dc984248833a75a725324d68
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [21]:
print(f'среднее для всех фолдов {np.mean(best_scores_for_folds)},стандартное отклоненние {np.std(best_scores_for_folds)}')

среднее для всех фолдов 0.17363838074062934,стандартное отклоненние 0.02901065445690522


## Обучение финальной модели

In [22]:
#Hyperparams
FINAL_EPOCHS = int(np.median(best_epochs_for_folds))
#Model
emb_din = 12
inner_n = 128
drop_n = 0.2
n_layers = 2
use_batchnorm = False
#lr_scheduler
factor = 0.9       # Коэффициент уменьшения LR: новый LR = старый LR * factor
patience = 10      # Количество эпох без значимого улучшения метрики до снижения LR
threshold = 0.01 #abs  # Минимальное изменение метрики, считающееся улучшением
min_lr = 0.00000001    # Нижняя граница LR, ниже которой scheduler его не уменьшает
#EarlyStopping
early_stopping_patience = 60 #Кол-во эпох без измененния отслеживаемой переменной
early_stopping_threshold = 0.00001 #Насколько должна измениться отслеживаемая переменная
#Adam
optim_lr = 0.005 #скорость 0.4 может приводить к тому, что оптимизатор проскакивает хорошие области функции потерь, значения предсказаний резко меняются, а в дальнейшем обучение застревает на неудачном плато.
#Dataloader
batch_size = 32 #Размер батча
#Saving threshold 
saving_threshold = 0.001

### Подбираем LR как медиану по CV

In [23]:
final_lr_schedule = []

for epoch in range(FINAL_EPOCHS):

    lr_values = [
        fold_lr[epoch]
        for fold_lr in learning_rate_all_folds
        if epoch < len(fold_lr)
    ]

    final_lr_schedule.append(
        float(np.median(lr_values))
    )

In [24]:
#Metrics
train_loss_all_epochs = []
train_RMSE_all_epochs = []
learning_rate_all_epochs = []

#------

with mlflow.start_run(run_name='Final_Embedding_MLP') as final_run:

    mlflow.log_params({
        'epochs': FINAL_EPOCHS,
        'emb_dim': emb_din,
        'inner_n': inner_n,
        'dropout': drop_n,
        'n_layers': n_layers,
        'use_batchnorm': use_batchnorm,
        'optim_lr': optim_lr,
        'batch_size': batch_size,
        'lr_scheduler_factor': factor,
        'lr_scheduler_patience': patience,
        'lr_scheduler_threshold': threshold,
        'lr_scheduler_min_lr': min_lr,
        'seed': SEED
    })

    #Преобразование данных
    train_loader,_,n_num,cat_num,encoder,scaler = transform_data(X, X, y, y, cat_cols, num_cols, batch_size)

    #Конфигурация модели
    model_reg,loss_reg,opt_reg,lr_scheduler,early_stopping = conf_model(n_num,cat_num,emb_din,inner_n,optim_lr,drop_n,factor,patience,threshold,min_lr,early_stopping_patience,early_stopping_threshold,n_layers,use_batchnorm)

    for epoch in tqdm(range(FINAL_EPOCHS), desc='Final Model Training'):

        current_lr = final_lr_schedule[epoch]

        for param_group in opt_reg.param_groups:
            param_group['lr'] = current_lr

        train_loss, train_RMSE, _, _ = run_epoch(model_reg,train_loader,None,loss_reg,opt_reg,flag='train')

        train_mse = train_loss[0]
        train_rmse = train_RMSE[0]

        train_loss_all_epochs.append(train_mse)
        train_RMSE_all_epochs.append(train_rmse)

        mlflow.log_metrics({
            'train_mse': train_mse,
            'train_rmse': train_rmse,
            'learning_rate': current_lr
        }, step=epoch + 1)

    #Сохранение финальной модели
    checkpoint_path = './DL_Models/final_pipeline.pt'

    model_config = {
    'cat_num': cat_num,
    'num_num': n_num,
    'emb_din': emb_din,
    'inner_n': inner_n,
    'drop_n': drop_n,
    'n_layers': n_layers,
    'use_batchnorm': use_batchnorm
    }

    torch.save({
        'model_state_dict': model_reg.state_dict(),
        'model_config': model_config,
        'encoder': encoder,
        'scaler': scaler,
        'cat_cols': cat_cols,
        'num_cols': num_cols
    }, checkpoint_path)

    mlflow.log_artifact(
        checkpoint_path,
        artifact_path='checkpoint'
    )

    mlflow.log_metrics({
        'final_train_rmse': train_RMSE_all_epochs[-1],
        'final_train_mse': train_loss_all_epochs[-1],
        'epochs_trained': FINAL_EPOCHS
    })

    mlflow.log_params({
        'n_numeric_features': n_num,
        'n_categorical_features': len(cat_num),
        'n_model_parameters': sum(
            p.numel() for p in model_reg.parameters()
        )
    })

    print(f'Final Train RMSE: {train_RMSE_all_epochs[-1]:.5f}')

Final Model Training:   0%|          | 0/215 [00:00<?, ?it/s]

🏃 View run Final_Embedding_MLP at: http://127.0.0.1:5000/#/experiments/3/runs/62df2b39b19a46f7bac68796673b3b4e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


KeyboardInterrupt: 